In [1]:
print("setup is ready")

setup is ready


In [2]:
%pwd

'c:\\AryanCollege\\Final_project\\resources'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\AryanCollege\\Final_project'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data, 
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [10]:
extracted_data = load_pdf_files("data")
print(extracted_data)


[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': 'data\\Aryandocument.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Aryan  college  of  Engineering  and  Management   College  Name  :  Aryan  college  of  Engineering  and  Management  University  affiliation:  Purbanchal  University  Location  :  Devkota  sadak,  Mid  -  Baneshower  ,  Kathmandu,  Nepal  Contact  NUmber:  01-4585146,  01-4582217  Gmail  :  info@thearyanschool.edu.np  Website:  https://thearyanschool.edu.np  Level:  Bachelors  Faculties  :  Bachelors  in  Information  Technology,  Bachelors  in  Computer  Application,  B.  Tech  in  \nAI,\n \nBE\n \nin\n \nEngineering,\n \nBE\n \nCivil\n \nEngineering\n \n principle  :  Er.  Dev  Raj  joshi  Vice  principle(Academic)  :  Mr.  prakash  Neupane  Vice-  principle(Admin  and  Accountant)  :  Mr.  Omkar  Nath  Gupta  Academic  Director:  Er.  Prakash  D

In [12]:
extracted_data
print(len(extracted_data))

6


In [15]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs (docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
            page_content=doc.page_content,
            metadata={"source": src}
            )
        )
     
    return minimal_docs

In [16]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [17]:
minimal_docs

[Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='Aryan  college  of  Engineering  and  Management   College  Name  :  Aryan  college  of  Engineering  and  Management  University  affiliation:  Purbanchal  University  Location  :  Devkota  sadak,  Mid  -  Baneshower  ,  Kathmandu,  Nepal  Contact  NUmber:  01-4585146,  01-4582217  Gmail  :  info@thearyanschool.edu.np  Website:  https://thearyanschool.edu.np  Level:  Bachelors  Faculties  :  Bachelors  in  Information  Technology,  Bachelors  in  Computer  Application,  B.  Tech  in  \nAI,\n \nBE\n \nin\n \nEngineering,\n \nBE\n \nCivil\n \nEngineering\n \n principle  :  Er.  Dev  Raj  joshi  Vice  principle(Academic)  :  Mr.  prakash  Neupane  Vice-  principle(Admin  and  Accountant)  :  Mr.  Omkar  Nath  Gupta  Academic  Director:  Er.  Prakash  Dahal  Finance  Head  :  Mrs.  Kabita  Adhikari  Account  Officer  :  Mrs.  Roji  Rani  Dev  Exam  officer  :  Mrs.  Dimple  shrestha  Librarian  :  Mrs.  Gita  Adhikari

In [22]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=10, 
        chunk_overlap=5
        )
    texts_chunks = text_splitter.split_documents(minimal_docs)
    return texts_chunks


In [25]:
texts_chunks = text_split(minimal_docs)
print(f"number of chunks: {len(texts_chunks)}")


number of chunks: 962


In [26]:
texts_chunks

[Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='Aryan'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='college'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='of'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='Engineeri'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='neering'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='and'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='Managemen'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='gement'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='College'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='Name  :'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content=':  Aryan'),
 Document(metadata={'source': 'data\\Aryandocument.pdf'}, page_content='college'),
 Document(m

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name,
        )
    return embeddings
embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
vector = embedding.embed_query("Hello world!")
vector

In [31]:
print("vector length:", len(vector))

vector length: 384


In [32]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [34]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [35]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key = pinecone_api_key )

In [36]:
pc

In [44]:
from pinecone import ServerlessSpec

index_name = "final-project"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud = "aws", region ="us-east-1")
    )
    
    
index = pc.Index(index_name)

In [45]:
from langchain_pinecone import PineconeVectorStore
docSearch = PineconeVectorStore.from_documents(
    documents = texts_chunks,
    embedding = embedding,
    index_name = index_name
)

In [47]:
from langchain_pinecone import PineconeVectorStore
docSearch = PineconeVectorStore.from_existing_index(
    index_name= index_name,
    embedding=embedding
)

In [48]:
realida =Document(
    page_content="This is a test document.",
    metadata={"source": "test.pdf"}
)

In [49]:
docSearch .add_documents(documents=[realida])

['354be006-2aa0-4678-af92-0a7fb482392b']

1.48